# Hypothesis Testing & Statistical Inference

This notebook runs a set of classical statistical tests on the defect-rate distribution and supplier quality differences.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from pathlib import Path

root = Path.cwd()
df = pd.read_csv(root / 'data' / 'analytical_dataset.csv')

df['production_date'] = pd.to_datetime(df['production_date'])
print(df[['defect_rate', 'quantity_produced', 'lead_time_days', 'quality_score']].describe().to_string())


In [ ]:
# Top risk and low-risk suppliers
ranked = df.groupby('supplier_id').agg(defect_rate=('defect_rate','mean')).reset_index().sort_values('defect_rate', ascending=False)
print('Top 10 high-risk suppliers:
', ranked.head(10).to_string(index=False))
print('Top 10 low-risk suppliers:
', ranked.tail(10).sort_values('defect_rate', ascending=False).to_string(index=False))


In [ ]:
# Test 1: independent t-test between two suppliers
# Choose suppliers with largest spread in mean defect rate
a = ranked.iloc[0]['supplier_id']
b = ranked.iloc[-1]['supplier_id']
A = df[df['supplier_id'] == a]['defect_rate']
B = df[df['supplier_id'] == b]['defect_rate']
result = stats.ttest_ind(A, B, equal_var=False)
print('Supplier A:', a)
print('Supplier B:', b)
print('t-statistic:', result.statistic)
print('p-value:', result.pvalue)
print('Interpretation:', 'Significant difference' if result.pvalue < 0.05 else 'No significant difference')


In [ ]:
# Test 2: ANOVA across all suppliers
groups = [group['defect_rate'].values for _, group in df.groupby('supplier_id')]
F, p = stats.f_oneway(*groups)
print('ANOVA F-statistic:', F)
print('ANOVA p-value:', p)
print('Interpretation:', 'Significant supplier differences' if p < 0.05 else 'No significant supplier differences')


In [ ]:
# Test 3: correlation analysis
cols = ['quantity_produced', 'lead_time_days', 'quality_score', 'defect_rate', 'days_since_start', 'batch_age_days']
correlation_matrix = df[cols].corr()
print(correlation_matrix.round(4).to_string())

plt.figure(figsize=(9, 7))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()


In [ ]:
# Test 4: normality test and log transform
stat, p = stats.shapiro(df['defect_rate'].dropna())
print('Shapiro-Wilk statistic:', stat)
print('Shapiro-Wilk p-value:', p)
print('Interpretation:', 'Not normal' if p < 0.05 else 'Approximately normal')

log_defect_rate = np.log1p(df['defect_rate'].clip(lower=1e-6))
stat_log, p_log = stats.shapiro(log_defect_rate.dropna())
print('Log transformed Shapiro-Wilk statistic:', stat_log)
print('Log transformed p-value:', p_log)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['defect_rate'], ax=axes[0], bins=30, kde=True)
axes[0].set_title('Raw Defect Rate Distribution')
sns.histplot(log_defect_rate, ax=axes[1], bins=30, kde=True, color='darkorange')
axes[1].set_title('Log-Transformed Defect Rate Distribution')
plt.tight_layout()


In [ ]:
# Test 5: chi-square test for defect type vs supplier
# Create a simple contingency table using top 5 suppliers vs defect categories
# This is intended to test association between supplier and defect type.
contingency = pd.crosstab(df['supplier_id'].map(lambda x: 'Top' if x <= 5 else 'Other'), defects['defect_type'].head(100).reindex(range(len(defects))).fillna('Unknown'))
# Use the actual defect table to avoid empty table creation
# A more meaningful test: compare defect categories across top suppliers by defect type counts
subset = defects[defects['defect_type'].isin(defects['defect_type'].value_counts().head(5).index)].copy()
contingency = pd.crosstab(subset['defect_type'], subset['severity'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print('Chi-square statistic:', chi2)
print('Chi-square p-value:', p)
print('Interpretation:', 'Significant association' if p < 0.05 else 'No significant association')

print(contingency)


In [ ]:
# Violin plot by supplier (top 10)
top10 = df.groupby('supplier_id')['defect_rate'].mean().sort_values(ascending=False).head(10).index
subset = df[df['supplier_id'].isin(top10)]
plt.figure(figsize=(12, 6))
sns.violinplot(data=subset, x='supplier_id', y='defect_rate', palette='viridis')
plt.title('Defect Rate by Supplier (Top 10 High-Risk)')
plt.xlabel('Supplier ID')
plt.ylabel('Defect Rate')
plt.xticks(rotation=45)
plt.tight_layout()
